# Funding Rate Arbitrage (Crypto Perpetuals)

**Strategy:**
- Entry: Funding rate > 0.05% (>55% APR) for 2+ periods
- Position: Long spot + Short perp (delta neutral)
- Exit: Funding rate crosses zero
- P&L: Collect funding payments

In [ ]:
import sys

sys.path.append('..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from medusa.strategies import FundingArbStrategy

%matplotlib inline

In [ ]:
# Generate mock funding rate data
np.random.seed(42)
n_periods = 365 * 3  # 3 funding periods per day (8h each) for 1 year
dates = pd.date_range('2024-01-01', periods=n_periods, freq='8h')

# Funding rate: mean-reverting process with occasional spikes
funding_mean = 0.0001  # 0.01% per 8h = ~10% APR baseline
funding_vol = 0.0003
funding_rate = np.random.normal(funding_mean, funding_vol, n_periods)

# Add occasional spikes (high funding events)
spike_mask = np.random.choice([0, 1], size=n_periods, p=[0.95, 0.05])
funding_rate += spike_mask * np.random.uniform(0.0003, 0.0010, n_periods)

# Smooth with moving average
funding_rate_smooth = pd.Series(funding_rate).rolling(3, min_periods=1).mean().values

# Mock spot and perp prices (mostly track each other)
spot_price = 100 + np.cumsum(np.random.randn(n_periods) * 0.5)
perp_price = spot_price + np.random.randn(n_periods) * 0.1  # Small basis

data = pd.DataFrame({
    'funding_rate': funding_rate_smooth,
    'close': spot_price,
    'spot_price': spot_price,
    'perp_price': perp_price,
}, index=dates)

print(f"Data shape: {data.shape}")
print("\nFunding rate statistics:")
print(f"  Mean: {data['funding_rate'].mean():.4%} per 8h")
print(f"  Std: {data['funding_rate'].std():.4%}")
print(f"  Max: {data['funding_rate'].max():.4%}")
print(f"  Min: {data['funding_rate'].min():.4%}")
data.head()

In [ ]:
# Calculate annualized funding rate
periods_per_year = (365 * 24) / 8
data['funding_apr'] = data['funding_rate'] * periods_per_year

print("\nAnnualized funding rate (APR):")
print(f"  Mean: {data['funding_apr'].mean():.1%}")
print(f"  Median: {data['funding_apr'].median():.1%}")
print(f"  Max: {data['funding_apr'].max():.1%}")

In [ ]:
# Visualize funding rate over time
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Funding rate (per 8h period)
axes[0].plot(data.index, data['funding_rate'], label='Funding Rate', color='purple', linewidth=1.5)
axes[0].axhline(0.0005, color='green', linestyle='--', label='Entry threshold (0.05%)')
axes[0].axhline(0, color='red', linestyle='-', label='Exit threshold (0%)')
axes[0].fill_between(data.index, 0.0005, data['funding_rate'].max(),
                      alpha=0.2, color='green', label='Arb zone')
axes[0].set_ylabel('Funding Rate (per 8h)')
axes[0].set_title('Funding Rate Chart')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Annualized APR
axes[1].plot(data.index, data['funding_apr'], label='APR', color='blue', linewidth=1.5)
axes[1].axhline(0.55, color='green', linestyle='--', label='Entry threshold (55% APR)')
axes[1].axhline(0, color='red', linestyle='-', label='Exit threshold (0% APR)')
axes[1].set_ylabel('Annualized Rate')
axes[1].set_xlabel('Date')
axes[1].set_title('Annualized Funding Rate (APR)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Initialize strategy
strategy = FundingArbStrategy(
    funding_entry_threshold=0.0005,  # 0.05% per 8h = 55% APR
    funding_exit_threshold=0.0,
    consecutive_periods=2,
    max_position_pct=0.5,
)

# Generate signals
entries, exits = strategy.generate_signals(data)

print(f"\nEntry signals: {entries.sum()}")
print(f"Exit signals: {exits.sum()}")
print(f"% Time in arbitrage: {entries.sum() / len(data):.1%}")

In [ ]:
# Simulate P&L from funding payments
# Assume we enter/exit based on signals
position = 0
pnl_series = []

for i in range(len(data)):
    if entries.iloc[i] and position == 0:
        position = 1  # Enter position
    elif exits.iloc[i] and position == 1:
        position = 0  # Exit position

    # Collect funding if in position (positive funding = longs pay shorts, we collect)
    funding_pnl = data['funding_rate'].iloc[i] * position
    pnl_series.append(funding_pnl)

data['funding_pnl'] = pnl_series
data['cumulative_pnl'] = data['funding_pnl'].cumsum()

print("\nFunding P&L:")
print(f"  Total return: {data['cumulative_pnl'].iloc[-1]:.2%}")
print(f"  Avg per period (when active): {data.loc[data['funding_pnl'] > 0, 'funding_pnl'].mean():.4%}")
print(f"  Annualized: {data['funding_pnl'].sum() * periods_per_year / len(data):.1%}")

In [ ]:
# Plot cumulative P&L
plt.figure(figsize=(14, 6))
plt.plot(data.index, data['cumulative_pnl'], label='Cumulative Funding P&L',
         color='green', linewidth=2)
plt.ylabel('Cumulative Return')
plt.xlabel('Date')
plt.title('Funding Arbitrage Cumulative P&L (Funding Payments Only)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Notes

This notebook simulates funding payment collection only. Full implementation requires:
1. Tracking both spot and perp positions
2. Calculating delta neutrality
3. Accounting for execution costs and basis risk
4. Real funding rate data from exchange APIs

Expected returns: 10-30% APR (market-neutral, low Sharpe but consistent carry)